# Mirai / Rises.io - CT inspection (milestone 1)

This notebook loads **one real CT at a time** and optionally overlays an **existing annotation**. It performs no inference and generates no predictions. Synthetic fixtures used by pytest are software checks, not patient results.

Select the project `.venv` / **Mirai CT (.venv)** kernel. Follow `README.md` to download the single public MSD sample and create the ignored `cases.local.csv` first. Never commit a notebook containing patient images, paths, IDs or outputs. Clear all outputs before committing; automated runs go to ignored `outputs/`.

Original geometry is preserved. The plots use native voxel planes and nearest anatomical axis direction labels, not standardized radiological views. Oblique or sheared acquisitions need additional expert inspection. Numerical label IDs are source annotations; their meaning must be checked against the dataset documentation.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from mirai_ct.cases import read_cases
from mirai_ct.visualization import plot_slice
from mirai_ct.volume import check_geometry, load_volume, memory_status, metadata

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from the Mirai project folder.")
print("Memory before loading (MiB):", memory_status())

## Select one case

Paths in the CSV are relative to the CSV directory, or absolute Windows paths. A blank `label_path` is valid. Use public pseudonymous IDs only. Set `MIRAI_CASES` and `MIRAI_CASE_ID` before launching the kernel to select another local manifest/case, or edit this cell. The notebook does not invent a case when files are missing.

In [ ]:
manifest = Path(os.environ.get("MIRAI_CASES", str(root / "cases.local.csv")))
if not manifest.is_file():
    raise FileNotFoundError("Create cases.local.csv using README.md; no real scan is bundled.")
cases = read_cases(manifest)
if not cases:
    raise ValueError("The case list is empty. Add one authorized real CT first.")
case_id = os.environ.get("MIRAI_CASE_ID", cases[0].case_id)
case = next((item for item in cases if item.case_id == case_id), None)
if case is None:
    raise ValueError("Requested case ID is not in the CSV.")
print("Case:", case.case_id, "| Source:", case.dataset_source)
scan = load_volume(case.scan_path)
mask = load_volume(case.label_path) if case.label_path else None
if mask is not None:
    check_geometry(scan, mask)
    print("Mask geometry matches; confirm source pairing independently.")
else:
    print("No annotation supplied; displaying CT only.")

## Inspect metadata

`shape` gives voxel counts; `spacing` gives voxel sizes in `spatial_units`. `orientation` describes increasing voxel axes in patient space (R/L, A/P, S/I). The affine maps voxel coordinates to physical coordinates. A geometry warning must be investigated; mismatched or ambiguous masks are blocked.

Estimated full float32 size is informational: this notebook never allocates that full array. NIfTI alone cannot establish contrast phase or HU calibration; check the dataset provenance.

In [ ]:
print(json.dumps(metadata(scan), indent=2))
print("Memory after header loading (MiB):", memory_status())

## Display slices and existing labels

Change `slices` to other native indices and rerun this cell. Default: three middle slices, one plane at a time. `MIRAI_SLICE_INDEX` optionally changes the axis-2 index for a reproducible run. Window/level defaults are 400/50 display intensity units; these do not change stored voxels. `.nii.gz` access may be slower because preceding compressed bytes are read again. Avoid animation or loading all slices on this laptop.

If there is a mask, the right image shows its existing nonzero labels. No label on a selected plane says nothing about the rest of the scan. Geometry agreement does not verify patient identity or annotation correctness.

In [ ]:
level, window = 50, 400
slices = [(axis, scan.shape[axis] // 2) for axis in range(3)]
if os.environ.get("MIRAI_SLICE_INDEX"):
    slices[2] = (2, int(os.environ["MIRAI_SLICE_INDEX"]))

for axis, index in slices:
    print(f"Axis {axis}, slice {index}: before", memory_status())
    fig = plot_slice(scan, axis=axis, index=index, mask=mask, level=level, window=window)
    display(fig)
    plt.close(fig)
    del fig
    print("After:", memory_status())
    assert not scan.in_memory
    assert mask is None or not mask.in_memory
print("Completed CT inspection only. No model was run.")

## Release the current case

Close figure references and release the volume proxies before selecting another case. This notebook is an engineering viewer, not a clinically validated diagnostic tool. Cancer detection, risk scores, model explanations and clinical performance remain untested.

In [ ]:
plt.close("all")
del scan, mask
print("Memory after releasing volume references (MiB):", memory_status())